# Stochastic Calculus from Zero · Episode 8: Changing the odds on a tree

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The coin world.** A share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses, eight possible worlds. Heads has a 60% real-world chance; a deposit grows 2% a step. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Two sets of odds · 4. The ratio Z · 5. Z averages to one · 6. Pricing with the real-world odds · 7. Z over time · 8. Part 1 checkpoint · 9. This episode's question · 10. Exercises · 11. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 8,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

A fair walk at +3 after 50 steps: the remaining 50 steps average 0, so the best guess after 100 is +3 (the walk is a martingale).

In [ ]:
out["prev"] = {"answer": 3}

## 3. Two sets of odds

The same 8 worlds carry two sets of weights: the real-world $P$ (heads 0.6) and the risk-neutral $Q$ (heads 0.55, Episode 6).

In [ ]:
cw = CoinWorld()
R = 1 + cw.rate
K = 100.0
call = lambda w: max(cw.price(w, 3) - K, 0.0)
Z = {w: cw.weight(w, "Q") / cw.weight(w, "P") for w in cw.worlds}
Zn = {n: cw.given(n, lambda w: Z[w]) for n in range(cw.steps + 1)}
rows = cw.table(Z=lambda w: Z[w], call=call, Z1=lambda w: Zn[1][w[:1]], Z2=lambda w: Zn[2][w[:2]])
out["coin"] = {"worlds": rows}
out["odds"] = {"p": cw.p, "q": cw.q, "P_HHH": rows[0]["P"], "Q_HHH": rows[0]["Q"], "P_TTT": rows[-1]["P"], "Q_TTT": rows[-1]["Q"]}
pd.DataFrame(rows)[["world", "P", "Q", "Z"]]

## 4. The ratio Z

$Z(\omega) = Q(\omega) / P(\omega)$, the **Radon-Nikodym derivative** of $Q$ with respect to $P$. Per toss it is $q/p = 0.917$ on heads and $(1-q)/(1-p) = 1.125$ on tails, and over three tosses the factors multiply.

In [ ]:
out["Z"] = {"up": cw.q / cw.p, "down": (1 - cw.q) / (1 - cw.p), "HHH": Z["HHH"], "TTT": Z["TTT"],
            "bad": [w for w in cw.worlds if Z[w] > 1], "good": [w for w in cw.worlds if Z[w] < 1]}
for w in cw.worlds:
    assert abs(Z[w] - out["Z"]["up"] ** w.count("H") * out["Z"]["down"] ** w.count("T")) < 1e-12
out["Z"]

## 5. Z averages to one

$E_P[Z] = \sum_\omega P(\omega) \frac{Q(\omega)}{P(\omega)} = \sum_\omega Q(\omega) = 1$. And for any random variable $X$: $E_Q[X] = E_P[Z X]$. Reweighting moves probability between worlds but never creates or destroys any.

In [ ]:
EZ = cw.expect(lambda w: Z[w])
assert abs(EZ - 1) < 1e-12
assert abs(cw.expect(lambda w: Z[w] * cw.price(w, 1)) - cw.expect(lambda w: cw.price(w, 1), "Q")) < 1e-9
out["EZ"] = {"value": EZ, "E_Q_S1": cw.expect(lambda w: cw.price(w, 1), "Q")}

## 6. Pricing with the real-world odds

Discounting the real-world average of the call's payoff gives the wrong price. Weight each world by $Z$ first and you get the risk-neutral price from Episode 6.

In [ ]:
naive = cw.expect(call) / R ** 3
via_Z = cw.expect(lambda w: Z[w] * call(w)) / R ** 3
q_price = cw.expect(call, "Q") / R ** 3
assert abs(via_Z - q_price) < 1e-12 and naive > via_Z
out["price"] = {"naive": naive, "via_Z": via_Z, "q_price": q_price, "strike": K,
                "rows": [{"label": "Real-world average, discounted", "value": naive},
                         {"label": "Weighted by Z, discounted", "value": via_Z}]}
out["price"]

## 7. Z over time

$Z_n = E_P[Z \mid \mathcal{F}_n]$ is the reweighting known at time $n$: $1$, then $0.917$ or $1.125$, and so on. It is a $P$-martingale. Its continuous-time version, $Z_t = \exp(-\theta W_t - \tfrac12\theta^2 t)$, is Girsanov's theorem (Part 3).

In [ ]:
for n in range(cw.steps):
    for b in Zn[n]:
        assert abs(Zn[n][b] - (cw.p * Zn[n + 1][b + "H"] + (1 - cw.p) * Zn[n + 1][b + "T"])) < 1e-12
out["Zn"] = {"Z0": Zn[0][""], "Z1": Zn[1], "Z2": Zn[2]}
out["Zn"]

## 8. Part 1 checkpoint

1. In the coin world, compute $E_P[S_2 \mid \mathcal{F}_1]$ in both blocks, then check the tower property.
2. Is $S_n / 1.04^n$ a martingale under $P$? Under $Q$?
3. For the world THT, give $P$, $Q$ and $Z$, and check $Q = Z \cdot P$.
4. A process has $E[X_n] = 0$ for all $n$. Does that make it a martingale? Give a counterexample.

Answers: (1) 124.8 and 83.2, averaging to 108.16 = $100 \times 1.04^2$. (2) Under $P$ yes (each toss multiplies by 1.04 on average); under $Q$ no (it multiplies by 1.02, so $S_n/1.04^n$ drifts down). (3) $P = 0.096$, $Q = 0.111375$, $Z = 1.1602$. (4) No: Episode 6's $X_2 = -X_1$.

In [ ]:
g = cw.given(1, lambda w: cw.price(w, 2))
assert abs(g["H"] - 124.8) < 1e-9 and abs(g["T"] - 83.2) < 1e-9
w = "THT"
assert abs(cw.weight(w, "Q") - Z[w] * cw.weight(w, "P")) < 1e-15
out["checkpoint"] = {"g_H": g["H"], "g_T": g["T"], "P_THT": cw.weight(w, "P"), "Q_THT": cw.weight(w, "Q"), "Z_THT": Z[w]}
out["checkpoint"]

## 9. This episode's question

If $Z = 1$ in every world, then $Q(\omega) = P(\omega)$ everywhere: the real-world and risk-neutral odds agree, which happens when $p = q$, i.e. when the share is expected to grow at exactly the deposit rate (no price of risk).

In [ ]:
same = CoinWorld(p=cw.q)
assert all(abs(same.weight(w, "Q") / same.weight(w, "P") - 1) < 1e-12 for w in same.worlds)
out["question"] = {"p_equals_q": cw.q}

## 10. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Compute $Z$ for the world HTH.

<details><summary>Answer</summary>

$Z = 0.917 \times 1.125 \times 0.917 = (0.55/0.6)^2 (0.45/0.4) = 0.9453$.

</details>

**Exercise 2 (By hand).** Check that $E_P[Z \cdot S_1] = E_Q[S_1] = 102$.

<details><summary>Answer</summary>

$Z_1$ is 0.917 on H and 1.125 on T: $0.6 \times 0.917 \times 120 + 0.4 \times 1.125 \times 80 = 66 + 36 = 102 = 1.02 \times 100$. ✓

</details>

**Exercise 3 (By hand).** What is $Z$ when $p = q$? What would it mean for the share's expected growth?

<details><summary>Answer</summary>

$Z = 1$ in every world. Then the real-world odds already make the discounted share a martingale: the share is expected to grow at the deposit rate, so investors are paid nothing extra for its risk.

</details>

**Exercise 4 (In code).** Build $Z$ for a 10-toss world. What are its smallest and largest values, and its standard deviation under $P$?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
cw10 = CoinWorld(steps=10)
z10 = np.array([cw10.weight(w, "Q") / cw10.weight(w, "P") for w in cw10.worlds])
p10 = np.array([cw10.weight(w, "P") for w in cw10.worlds])
assert abs(p10 @ z10 - 1) < 1e-12
sd = math.sqrt(p10 @ (z10 - 1) ** 2)
print(f"Z from {z10.min():.3f} to {z10.max():.3f}; sd under P {sd:.3f}")

**Exercise 5 (In code).** Draw a random payoff column $X$ (one number per world) and check $E_Q[X] = E_P[Z X]$.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
X = dict(zip(cw.worlds, np.random.default_rng(seed).normal(size=8)))
assert abs(cw.expect(lambda w: X[w], "Q") - cw.expect(lambda w: Z[w] * X[w])) < 1e-12
print("E_Q[X] = E_P[Z X] for a random payoff: checked")

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")